# 04: Train MLM (Masked Language Model)

In [1]:
!uv pip install --quiet tokenizers datasets transformers torch

## Dataset & Tokenizer

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    path="HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train",
    streaming=False,
).select(range(20000))

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/102 [00:00<?, ?it/s]

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE


SPECIAL_TOKENS = [
    "[_UNK_]",  # unknown token
    "[_PAD_]",  # padding token
    "[_INIT_]", # init token
    "[_STOP_]"  # stop token
]

tokenizer = Tokenizer(BPE(unk_token="[_UNK_]"))

In [4]:
from tokenizers.trainers import BpeTrainer

trainer = BpeTrainer(special_tokens=SPECIAL_TOKENS)

In [5]:
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

In [6]:
tokenizer.train_from_iterator(dataset["text"], trainer)

## MLM DataLoader

In [7]:
from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

MAX_LENGTH = 128

# Wrap the tokenizers.Tokenizer for HuggingFace compatibility
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[_UNK_]",
    pad_token="[_PAD_]",
    bos_token="[_INIT_]",
    eos_token="[_STOP_]",
)
fast_tokenizer.add_special_tokens({"mask_token": "[MASK]"})

# Tokenize dataset
def tokenize_fn(batch):
    return fast_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")

# MLM collator + DataLoader
mlm_collator = DataCollatorForLanguageModeling(tokenizer=fast_tokenizer, mlm=True, mlm_probability=0.15)
mlm_loader = DataLoader(tokenized_dataset, batch_size=128, shuffle=True, collate_fn=mlm_collator)

## Model — Transformer Encoder

A minimal BERT-style encoder built entirely in pure PyTorch:

```
Input IDs
   └─► Embedding (token + position)
         └─► TransformerEncoderLayer × N
               └─► MLM Head  (Linear → GELU → LayerNorm → Linear)
                     └─► logits  [batch, seq_len, vocab_size]
```

In [8]:
import torch
import torch.nn as nn


class MLMModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=fast_tokenizer.pad_token_id)
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)
        self.dropout   = nn.Dropout(dropout)

        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder   = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # MLM head
        self.mlm_dense = nn.Linear(d_model, d_model)
        self.mlm_act   = nn.GELU()
        self.mlm_norm  = nn.LayerNorm(d_model)
        self.mlm_proj  = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)

        x = self.dropout(self.token_emb(input_ids) + self.pos_emb(positions))

        # TransformerEncoder expects src_key_padding_mask: True = ignore
        pad_mask = (attention_mask == 0) if attention_mask is not None else None
        x = self.encoder(x, src_key_padding_mask=pad_mask)

        x = self.mlm_norm(self.mlm_act(self.mlm_dense(x)))
        return self.mlm_proj(x)  # (B, T, vocab_size)


VOCAB_SIZE = fast_tokenizer.vocab_size + len(fast_tokenizer.added_tokens_encoder)
model = MLMModel(vocab_size=VOCAB_SIZE)
print(f"Vocab size : {VOCAB_SIZE}")
print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}")

Vocab size : 30005
Parameters : 18,650,677


## Training Loop

In [ ]:
import tqdm

EPOCHS    = 20
LR        = 1e-4
DEVICE    = "cpu"


if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"


model     = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # -100 = non-masked positions

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    
    for bx, batch in tqdm.tqdm(enumerate(mlm_loader), ncols=80, total=len(mlm_loader)):
        
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        logits = model(input_ids, attention_mask)   # (B, T, V)

        # CrossEntropyLoss expects (N, C) and (N,)
        loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    # print loss after epoch:
    print(f"Epoch {epoch:>3}/{EPOCHS}  loss: {epoch_loss / len(mlm_loader):.4f}")

100%|█████████████████████████████████████████| 157/157 [08:42<00:00,  3.33s/it]


Epoch   1/20  loss: 8.6549


100%|█████████████████████████████████████████| 157/157 [09:44<00:00,  3.72s/it]


Epoch   2/20  loss: 7.7033


100%|█████████████████████████████████████████| 157/157 [09:21<00:00,  3.57s/it]


...

100%|█████████████████████████████████████████| 157/157 [07:36<00:00,  2.91s/it]


Epoch   6/20  loss: 7.5171


100%|█████████████████████████████████████████| 157/157 [09:22<00:00,  3.59s/it]


Epoch   7/20  loss: 7.4888


 38%|████████████████                          | 60/157 [03:12<05:34,  3.44s/it]

## Inference — Fill-Mask

In [25]:
model.eval()
mask_id = fast_tokenizer.mask_token_id

test_sentences = [
    "I [MASK] playing cricket.",
    "You can't [MASK] my mobile number.",
]

with torch.no_grad():
    for sentence in test_sentences:
        enc = fast_tokenizer(sentence, return_tensors="pt").to(DEVICE)
        logits = model(enc["input_ids"], enc["attention_mask"])  # (1, T, V)

        mask_positions = (enc["input_ids"] == mask_id).nonzero(as_tuple=True)[1]
        for pos in mask_positions:
            predicted_id    = logits[0, pos].argmax(-1).item()
            predicted_token = fast_tokenizer.convert_ids_to_tokens([predicted_id])[0]
            print(f"Input     : {sentence}")
            print(f"Predicted : {predicted_token}")
            print()

Input     : I [MASK] playing cricket.
Predicted : Ċ

Input     : You can't [MASK] my mobile number.
Predicted : Ċ

